# 飞书 (Feishu/Lark) API 全流程验证

> **目标**: 验证飞书 Open API 文档/知识库/权限全流程
> **覆盖**: 创建、读取、追加(文本/代码/公式/表格)、更新、删除、搜索、分享、Wiki 知识库
> **鉴权**: App ID + App Secret → tenant_access_token, user_access_token (Wiki 必须)
> **测试手机号**: `13586820267`

In [1]:
import os
import sys
import json
import time
from datetime import datetime
from pathlib import Path

# 将当前目录加入 Python 路径,以便导入同目录的 feishu_client
sys.path.insert(0, str(Path.cwd()))

# 加载 .env 文件
project_root = Path.cwd()
while project_root.name != "MetaBlog" and project_root.parent != project_root:
    project_root = project_root.parent

env_path = project_root / ".env"
if env_path.exists():
    try:
        from dotenv import load_dotenv
        load_dotenv(env_path)
        print(f"[OK] Loaded .env from {env_path}")
    except ImportError:
        with open(env_path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    key, value = line.split("=", 1)
                    os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
        print(f"[OK] Loaded .env from {env_path} (manual parse)")
else:
    print(f"[WARN] .env not found at {env_path}, using existing environment variables")

# 导入 FeishuClient
from feishu_client import FeishuClient, md_to_blocks

# 初始化客户端
client = FeishuClient()

# 健康检查
health = client.health_check()
print(f"[{'OK' if health['ok'] else 'FAIL'}] FeishuClient initialized")
if health['ok']:
    print(f"       Token valid: {health['token_valid']}")
    print(f"       Expire in: {health['expire_in']}s")
else:
    print(f"       Error: {health.get('error', 'Unknown')}")

# 测试手机号(用于用户搜索和权限分享)
TEST_MOBILE = "13586820267"

# 全局变量(后续 cell 共用)
DOC_A_ID = None
DOC_B_ID = None
DOC_A_TITLE = None
DOC_B_TITLE = None
WIKI_SPACE_ID = None
WIKI_NODE_A = None
WIKI_NODE_B = None
TEST_USER_OPEN_ID = None

# 飞书域名前缀(用于生成可点击链接)
FEISHU_DOMAIN = "https://feishu.cn"  # 根据你的租户自定义域名修改


[OK] Loaded .env from d:\ALL IN AI\MetaBlog\.env
[OK] FeishuClient initialized
       Token valid: True
       Expire in: 6397s


d:\python-envs\baseenv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## 🔑 获取 user_access_token(Wiki 必需)

> **前提**：在飞书开放平台 → 你的应用 → **安全设置** → 添加重定向 URL: `http://localhost:8088`
> **作用**：Wiki 相关 API(创建空间、移动节点等)必须使用 user_access_token. 
> **说明**：如果 `.env` 中已有有效的 `FEISHU_USER_ACCESS_TOKEN`,可跳过此 cell. 

In [2]:
# 一键获取 user_access_token(OAuth 授权流程)

# 检查是否已有有效 token
existing_token = os.environ.get("FEISHU_USER_ACCESS_TOKEN", "")
print(f"[INFO] 当前 FEISHU_USER_ACCESS_TOKEN: {existing_token}...") if existing_token else print("[INFO] 当前未配置 FEISHU_USER_ACCESS_TOKEN")
# existing_token = None
if existing_token:
    print(f"[INFO] 已配置 FEISHU_USER_ACCESS_TOKEN: {existing_token[:20]}...")
    print("       如需重新获取,请注释掉上面两行再运行")
else:
    print("[INFO] 未配置 FEISHU_USER_ACCESS_TOKEN,开始 OAuth 授权流程...")
    print("       请确保已在安全设置中添加重定向 URL: http://localhost:8088\n")
    try:
        token_info = client.authorize_user(redirect_uri="http://localhost:8088", port=8088, timeout=120)
        print(f"\n[OK] user_access_token 获取成功!")
        print(f"       Token: {token_info.get('access_token', '')[:20]}...")
        print(f"       Expires in: {token_info.get('expires_in', 'N/A')}s")
        print(f"       Scope: {token_info.get('scope', 'N/A')}")
        # 更新环境变量和客户端实例
        os.environ["FEISHU_USER_ACCESS_TOKEN"] = token_info["access_token"]
        client.user_access_token = token_info["access_token"]
    except Exception as e:
        print(f"\n[FAIL] 授权失败: {e}")
        print("\n排查建议:")
        print("  1. 确认已在安全设置中添加 http://localhost:8088")
        print("  2. 确认应用已发布或处于测试状态")
        print("  3. 手动复制浏览器地址栏中的授权链接,确保能正常打开")


[INFO] 当前 FEISHU_USER_ACCESS_TOKEN: eyJhbGciOiJFUzI1NiIsImZlYXR1cmVfY29kZSI6IkZlYXR1cmVPQXV0aEpXVFNpZ25fQ04iLCJraWQiOiI3NjMzOTY0MjE0NTkxMjQ1NTMwIiwidHlwIjoiSldUIn0.eyJqdGkiOiI3NjM1MzU4MDMzNzIzMjYzOTU2IiwiaWF0IjoxNzc3NzQ1MzI1LCJleHAiOjE3Nzc3NTI1MjUsInZlciI6InYxIiwidHlwIjoiYWNjZXNzX3Rva2VuIiwiY2xpZW50X2lkIjoiY2xpX2E5NmNmZjQ2OTNiODVjYzAiLCJzY29wZSI6ImF1dGg6dXNlci5pZDpyZWFkIG9mZmxpbmVfYWNjZXNzIiwiYXV0aF9pZCI6Ijc2MzUzNTgwMzExNzMxMjczNTkiLCJhdXRoX3RpbWUiOjE3Nzc3NDUzMjQsImF1dGhfZXhwIjoxODA5MjgxMzI0LCJ1bml0IjoiZXVfbmMiLCJ0ZW5hbnRfdW5pdCI6ImV1X25jIiwib3BhcXVlIjp0cnVlLCJlbmMiOiJBaVFrQVFFQ0FNSURBQUVCQXdBQ0FRMEFBd3NMQUFBQUF3QUFBQWRHWldGMGRYSmxBQUFBRUc5aGRYUm9YMjl3WVhGMVpWOXFkM1FBQUFBSVZHVnVZVzUwU1dRQUFBQUJNQUFBQUFSVWFXMWxBQUFBQ2pFM056Y3lORGd3TURBUEFBUU1BQUFBQVFvQUFXTEZJbG12Z0FBaUN3QUNBQUFBREZpUUk4ZWRkNnMrUXFJZjlnc0FBd0FBQURBdHFoZXVKeTlSWEtKd2R2OFlrcFpXV09yQUNqdE1NNEViK0szWitCRVBrcldwd3doa2tBUGs2R2NJWnEzdjQvb0FDd0FGQUFBQUJXVjFYMjVqQU1lcmxVbUdmM2doQzVBR0ZrSkcwNXVjOGpUWmplSWZXT1VheXkzWVM4UFpYRVlnbjhy

## 1. 用户搜索

通过手机号搜索飞书用户,获取 `open_id`(后续用于文档权限分享). 

In [3]:
# 用手机号搜索用户
print(f"[INFO] Searching user by mobile: {TEST_MOBILE}")
try:
    # 飞书 contact API 不支持直接按手机号搜索,使用 users/batch_get_id 转换
    # 先尝试用 users API 获取
    user_result = client.api("POST", "/contact/v3/users/batch_get_id", json_data={
        "mobiles": [TEST_MOBILE],
        "include_resigned": False
    })
    users = user_result.get("user_list", [])
    if users:
        user = users[0]
        TEST_USER_OPEN_ID = user.get("user_id", "")
        print(f"[OK] Found user:")
        print(f"       open_id: {TEST_USER_OPEN_ID}")
        print(f"       mobile: {user.get('mobile', 'N/A')}")
    else:
        print(f"[WARN] User not found with mobile {TEST_MOBILE}")
        print("       Will skip permission sharing tests")
except Exception as e:
    print(f"[WARN] User search failed: {e}")
    print("       Will skip permission sharing tests")


[INFO] Searching user by mobile: 13586820267
[OK] Found user:
       open_id: ou_5fd95247243d26b82a1e1a5fdefe973d
       mobile: 13586820267


## 2. 创建文档

创建两个测试文档：
- **doc_A**: 用于追加/更新/删除块演示(保留到最后)
- **doc_B**: 用于删除演示(最后删除)

In [4]:
# 创建 doc_A
title_a = f"API验证-A-{datetime.now().strftime('%H:%M:%S')}"
try:
    result_a = client.api("POST", "/docx/v1/documents", json_data={"title": title_a})
    DOC_A_ID = result_a["document"]["document_id"]
    DOC_A_TITLE = title_a
    print(f"[OK] Created doc_A: {DOC_A_TITLE}")
    print(f"     document_id: {DOC_A_ID}")
    print(f"     URL: https://feishu.cn/docx/{DOC_A_ID}")
except Exception as e:
    print(f"[FAIL] Create doc_A failed: {e}")
    raise

# 创建 doc_B
title_b = f"API验证-B-{datetime.now().strftime('%H:%M:%S')}"
try:
    result_b = client.api("POST", "/docx/v1/documents", json_data={"title": title_b})
    DOC_B_ID = result_b["document"]["document_id"]
    DOC_B_TITLE = title_b
    print(f"[OK] Created doc_B: {DOC_B_TITLE}")
    print(f"     document_id: {DOC_B_ID}")
except Exception as e:
    print(f"[FAIL] Create doc_B failed: {e}")
    raise


[OK] Created doc_A: API验证-A-22:56:57
     document_id: ZZSMdJTRgo9YxDxGJTlcVIWzn1b
     URL: https://feishu.cn/docx/ZZSMdJTRgo9YxDxGJTlcVIWzn1b
[OK] Created doc_B: API验证-B-22:56:58
     document_id: Arhdda0x3oXdSHxt5cGcfzV9nSb


## 3. 读取文档(初始状态)

读取 doc_A 和 doc_B 的原始内容和块结构,记录初始状态. 

In [5]:
def print_doc_summary(doc_id, label):
    """打印文档摘要"""
    try:
        # 读取元数据
        meta = client.api("GET", f"/docx/v1/documents/{doc_id}")
        print(f"\n[{label}] {meta['document']['title']}")
        print(f"       document_id: {doc_id}")
        # 读取块结构
        blocks = client.get_doc_blocks(doc_id)
        items = blocks.get("items", [])
        print(f"       Total blocks: {len(items)}")
        for item in items[:3]:
            bt = item.get("block_type", "?")
            print(f"       - block_type={bt}, block_id={item.get('block_id', '?')[:20]}...")
        if len(items) > 3:
            print(f"       ... and {len(items)-3} more")
    except Exception as e:
        print(f"[WARN] Read {label} failed: {e}")

print_doc_summary(DOC_A_ID, "doc_A INITIAL")
print_doc_summary(DOC_B_ID, "doc_B INITIAL")



[doc_A INITIAL] API验证-A-22:56:57
       document_id: ZZSMdJTRgo9YxDxGJTlcVIWzn1b
       Total blocks: 1
       - block_type=1, block_id=ZZSMdJTRgo9YxDxGJTlc...

[doc_B INITIAL] API验证-B-22:56:58
       document_id: Arhdda0x3oXdSHxt5cGcfzV9nSb
       Total blocks: 1
       - block_type=1, block_id=Arhdda0x3oXdSHxt5cGc...


## 4. 追加文本内容

向 doc_A 追加 Markdown 文本：标题、段落、无序列表、加粗. 使用 `md_to_blocks` 转换. 

In [6]:
text_md = """## 追加文本测试

这是**一段加粗的文本**,用于验证行内格式. 

### 列表测试
- 无序列表项 1
- 无序列表项 2
- 带有 `inline code` 的列表项

> 这是一个引用块,用于验证引用格式. 
"""

try:
    blocks = md_to_blocks(text_md)
    result = client.api("POST", f"/docx/v1/documents/{DOC_A_ID}/blocks/{DOC_A_ID}/children",
                        json_data={"children": blocks})
    children = result.get("children", [])
    print(f"[OK] Appended {len(children)} text blocks to doc_A")
except Exception as e:
    print(f"[FAIL] Append text failed: {e}")


[OK] Appended 7 text blocks to doc_A


## 5. 读取文档(追加文本后)→ 对比

读取 doc_A 的块结构,与初始状态对比. 

In [7]:
print(f"\n[doc_A AFTER TEXT APPEND]")
try:
    blocks = client.get_doc_blocks(DOC_A_ID)
    items = blocks.get("items", [])
    print(f"       Total blocks: {len(items)}")
    print(f"       [CHANGE] +{len(items) - 1} blocks (from 1 empty doc to {len(items)} blocks)")
    for item in items[-3:]:
        bt = item.get("block_type", "?")
        print(f"       - block_type={bt}")
except Exception as e:
    print(f"[WARN] Read failed: {e}")



[doc_A AFTER TEXT APPEND]
       Total blocks: 8
       [CHANGE] +7 blocks (from 1 empty doc to 8 blocks)
       - block_type=12
       - block_type=12
       - block_type=15


## 6. 追加代码块

向 doc_A 追加 fenced code block(Python 语言标识). 

In [8]:
code_md = """
```python
def hello_opd():
    print("Hello OPD!")
    return "success"

# 调用
result = hello_opd()
```
"""

try:
    blocks = md_to_blocks(code_md)
    result = client.api("POST", f"/docx/v1/documents/{DOC_A_ID}/blocks/{DOC_A_ID}/children",
                        json_data={"children": blocks})
    print(f"[OK] Appended code block to doc_A")
except Exception as e:
    print(f"[FAIL] Append code failed: {e}")


[OK] Appended code block to doc_A


## 7. 读取文档(追加代码后)→ 对比

In [9]:
print(f"\n[doc_A AFTER CODE APPEND]")
try:
    blocks = client.get_doc_blocks(DOC_A_ID)
    items = blocks.get("items", [])
    code_blocks = [i for i in items if i.get("block_type") == 14]
    print(f"       Total blocks: {len(items)}")
    print(f"       Code blocks: {len(code_blocks)}")
    if code_blocks:
        lang = code_blocks[0].get("code", {}).get("style", {}).get("language", "unknown")
        print(f"       Code language: {lang}")
except Exception as e:
    print(f"[WARN] Read failed: {e}")



[doc_A AFTER CODE APPEND]
       Total blocks: 9
       Code blocks: 1
       Code language: 49


## 8. 追加数学公式

向 doc_A 追加行内公式 `$...$` 和块级公式 `$$...$$`. 

In [10]:
formula_md = """## 公式测试

行内公式：$\mathcal{L}_{\text{OPD}} = \mathbb{E}_{x \sim \mathcal{D}}[\text{KL}(\pi_T(\cdot|x) \| \pi_\theta(\cdot|x))]$

块级公式：
$$\nabla_\theta J(\theta) = \mathbb{E}_{a \sim \pi_\theta}[\nabla_\theta \log \pi_\theta(a|s) \cdot A(s,a)]$$

以及多行对齐：
$$\begin{aligned}
Q(y|x) &= \beta \cdot \log \pi_{\text{ref}}(y|x) \\
\tau^{(t)} &= \tau_0 \cdot \exp(-\alpha t)
\end{aligned}$$
"""

try:
    blocks = md_to_blocks(formula_md)
    result = client.api("POST", f"/docx/v1/documents/{DOC_A_ID}/blocks/{DOC_A_ID}/children",
                        json_data={"children": blocks})
    print(f"[OK] Appended formula blocks to doc_A")
except Exception as e:
    print(f"[FAIL] Append formula failed: {e}")


[OK] Appended formula blocks to doc_A


## 9. 读取文档(追加公式后)→ 对比

In [11]:
print(f"\n[doc_A AFTER FORMULA APPEND]")
try:
    blocks = client.get_doc_blocks(DOC_A_ID)
    items = blocks.get("items", [])
    # 统计 equation 元素
    eq_count = 0
    for item in items:
        els = item.get("text", {}).get("elements", [])
        for el in els:
            if el.get("equation"):
                eq_count += 1
    print(f"       Total blocks: {len(items)}")
    print(f"       Equation elements: {eq_count}")
except Exception as e:
    print(f"[WARN] Read failed: {e}")



[doc_A AFTER FORMULA APPEND]
       Total blocks: 13
       Equation elements: 2


## 10. 追加表格

向 doc_A 追加 Markdown 表格(含公式列,验证表格内公式 `|` 保护). 

In [34]:
table_md = """## 表格测试

| 特性 | **MHA (Multi-Head Attention)** | **MQA (Multi-Query Attention)** | **GQA (Grouped-Query Attention)** | **MLA (Multi-head Latent Attention)** |
| --- | --- | --- | --- | --- |
| **核心思想** | 并行多子空间学习 | 极致KV共享,压缩**Cache** | 在性能和效率间折中 | 低秩压缩KV,推理时吸收权重 |
| **Q头数量** | $N_q\_heads$ | $N_q\_heads$ | $N_q\_heads$ | $N_q\_heads$ |
| **KV头数量** | $N_q\_heads$ | 1 | $N_{kv}\_heads$ ($1 \le N_{kv}\_heads \le N_q\_heads$) | 训练时为 $N_q\_heads$,推理时等效为1个共享**潜在**K/V |
| **KV Cache内容** | 独立的K, V向量 ($N_q\_heads \times d_{head}$) | 共享的K, V向量 ($1 \times d_{head}$) | 分组共享的K, V向量 ($N_{kv}\_heads \times d_{head}$) | 低维内容潜在向量 ($d_c$) + 低维RoPE键向量 ($d_r$) |
| **KV Cache大小 (相对MHA)** | 1x | $1/N_q\_heads$ | $N_{kv}\_heads / N_q\_heads$ | 极小 (与 $(d_c + d_r)$ 相关,通常远小于 $d_{model}$),约 $1/25$ 至 $1/60$ |
| **KV Cache节省原理** | - | 牺牲信息多样性,极致共享K,V | 平衡信息多样性与共享,分组K,V | 假设KV信息可低秩压缩,并用矩阵吸收技巧高效利用 |
| **计算效率变化** | $O(S^2)$(Prefill)/$O(S)$(Decoding) | 更快(Prefill),更快(Decoding) | 快(Prefill),更快(Decoding) | 训练时与MHA类似,推理时KV Cache极小但单token计算量可能略增 |
| **牺牲了什么** | 高昂的显存占用 | 模型性能(表达能力下降) | 组数超参调优,略有性能损失 | 模型结构和训练复杂性,训练稳定性挑战 |
"""

try:
    blocks = md_to_blocks(table_md)
    result = client.api("POST", f"/docx/v1/documents/{DOC_A_ID}/blocks/{DOC_A_ID}/children",
                        json_data={"children": blocks})
    table_block = next((c for c in result.get("children", []) if c.get("block_type") == 31), None)
    if table_block:
        prop = table_block.get("table", {}).get("property", {})
        print(f"[OK] Appended table to doc_A: {prop.get('row_size', '?')} rows x {prop.get('column_size', '?')} cols")
    else:
        print(f"[OK] Appended table blocks to doc_A")
except Exception as e:
    print(f"[FAIL] Append table failed: {e}")


[OK] Appended table to doc_A: 9 rows x 5 cols


## 11. 读取文档(追加表格后)→ 对比

In [13]:
print(f"\n[doc_A AFTER TABLE APPEND]")
try:
    blocks = client.get_doc_blocks(DOC_A_ID)
    items = blocks.get("items", [])
    tables = [i for i in items if i.get("block_type") == 31]
    print(f"       Total blocks: {len(items)}")
    print(f"       Table blocks: {len(tables)}")
    if tables:
        prop = tables[0].get("table", {}).get("property", {})
        print(f"       Table size: {prop.get('row_size', '?')} x {prop.get('column_size', '?')}")
except Exception as e:
    print(f"[WARN] Read failed: {e}")



[doc_A AFTER TABLE APPEND]
       Total blocks: 45
       Table blocks: 1
       Table size: 5 x 3


## 12. 更新指定块

获取 doc_A 的块结构,找到第一个文本块,更新其内容. 

In [14]:
print(f"\n[UPDATE BLOCK in doc_A]")
try:
    # 获取所有块
    blocks = client.get_doc_blocks(DOC_A_ID)
    items = blocks.get("items", [])

    # 找到第一个文本块(block_type=2)且不是根节点本身
    target = None
    for item in items:
        if item.get("block_type") == 2 and item.get("block_id") != DOC_A_ID:
            target = item
            break

    if not target:
        print("[WARN] No text block found to update")
    else:
        block_id = target["block_id"]
        old_content = target.get("text", {}).get("elements", [{}])[0].get("text_run", {}).get("content", "(empty)")
        print(f"       Target block_id: {block_id[:30]}...")
        print(f"       Old content: {old_content[:60]}...")

        # 更新块内容
        new_text = "【已更新】这是被 update_doc_block 修改后的内容,验证了 PATCH 接口的正确性. "
        update_data = {
            "update_text_elements": {
                "elements": [{"text_run": {"content": new_text}}]
            }
        }
        client.update_doc_block(DOC_A_ID, block_id, update_data)
        print(f"[OK] Block updated")
        print(f"       New content: {new_text[:60]}...")
except Exception as e:
    print(f"[FAIL] Update block failed: {e}")



[UPDATE BLOCK in doc_A]
       Target block_id: doxcn7Uslb0yng0z274O7kkoUAg...
       Old content: 这是...
[OK] Block updated
       New content: 【已更新】这是被 update_doc_block 修改后的内容,验证了 PATCH 接口的正确性. ...


## 13. 读取文档(更新块后)→ 对比

In [15]:
print(f"\n[doc_A AFTER BLOCK UPDATE]")
try:
    blocks = client.get_doc_blocks(DOC_A_ID)
    items = blocks.get("items", [])
    # 找到刚才更新的块(找包含"已更新"的文本块)
    for item in items:
        if item.get("block_type") == 2:
            content = item.get("text", {}).get("elements", [{}])[0].get("text_run", {}).get("content", "")
            if "已更新" in content:
                print(f"       [VERIFIED] Found updated block:")
                print(f"       Content: {content[:80]}...")
                break
    else:
        print(f"       [WARN] Updated block not found in current structure")
except Exception as e:
    print(f"[WARN] Read failed: {e}")



[doc_A AFTER BLOCK UPDATE]
       [VERIFIED] Found updated block:
       Content: 【已更新】这是被 update_doc_block 修改后的内容,验证了 PATCH 接口的正确性. ...


## 14. 删除指定块

获取 doc_A 的块结构,删除一个目标块(安全：只删非第一个文本块). 

In [16]:
print(f"\n[DELETE BLOCK in doc_A]")
try:
    blocks = client.get_doc_blocks(DOC_A_ID)
    items = blocks.get("items", [])

    # 找一个可删除的目标块(跳过根节点和第一个文本块)
    target = None
    text_count = 0
    for item in items:
        if item.get("block_type") == 2 and item.get("block_id") != DOC_A_ID:
            text_count += 1
            if text_count >= 2:  # 从第二个文本块开始删
                target = item
                break

    if not target:
        print("[SKIP] No suitable block found to delete")
    else:
        block_id = target["block_id"]
        old_content = target.get("text", {}).get("elements", [{}])[0].get("text_run", {}).get("content", "(empty)")
        print(f"       Target block_id: {block_id[:30]}...")
        print(f"       Content to delete: {old_content[:60]}...")

        # 删除
        client.delete_doc_block(DOC_A_ID, block_id)
        print(f"[OK] Block deleted")
except Exception as e:
    print(f"[FAIL] Delete block failed: {e}")



[DELETE BLOCK in doc_A]
       Target block_id: doxcn6iRbtU1GQ1XZXknC1AISke...
       Content to delete: 行内公式：...
[OK] Block deleted


## 15. 读取文档(删除块后)→ 对比

In [17]:
print(f"\n[doc_A AFTER BLOCK DELETE]")
try:
    blocks_before = len(items) if 'items' in dir() else 0  # 引用上一轮的 items
    blocks = client.get_doc_blocks(DOC_A_ID)
    items = blocks.get("items", [])
    print(f"       Total blocks: {len(items)}")
    print(f"       [CHANGE] -1 block deleted")
except Exception as e:
    print(f"[WARN] Read failed: {e}")



[doc_A AFTER BLOCK DELETE]
       Total blocks: 44
       [CHANGE] -1 block deleted


## 16. 删除文档(安全模式)

删除 doc_B,保留 doc_A. 
安全确认：只删除标题包含 "API验证" 的测试文档. 

In [18]:
print(f"\n[DELETE DOC]")
# 安全确认
if DOC_B_TITLE and "API验证" in DOC_B_TITLE:
    try:
        client.api("DELETE", f"/docx/v1/documents/{DOC_B_ID}")
        print(f"[OK] Deleted doc_B: {DOC_B_TITLE}")
        print(f"     document_id: {DOC_B_ID}")
    except Exception as e:
        print(f"[FAIL] Delete doc_B failed: {e}")
else:
    print(f"[SKIP] doc_B '{DOC_B_TITLE}' does not look like a test doc, skipping delete")
    print(f"       If you want to delete it anyway, run: client.api('DELETE', '/docx/v1/documents/{DOC_B_ID}')")



[DELETE DOC]
[OK] Deleted doc_B: API验证-B-22:56:58
     document_id: Arhdda0x3oXdSHxt5cGcfzV9nSb


## 17. 搜索文档

搜索刚才创建的文档. 

In [19]:
print(f"\n[SEARCH DOCS]")
try:
    result = client.search_docs("API验证")
    docs = result.get("docs_entities", [])
    print(f"[OK] Found {len(docs)} docs matching 'API验证'")
    for doc in docs[:3]:
        print(f"       - {doc.get('title', '?')} ({doc.get('docs_token', '?')[:20]}...)")
except Exception as e:
    print(f"[WARN] Search failed: {e}")



[SEARCH DOCS]
[OK] Found 20 docs matching 'API验证'
       - API验证-B-02:09:46 (C4p6d5nycoYavAxFh0cc...)
       - API验证-B-01:55:40 (CixhdNqpoo1W1DxMDl5c...)
       - API验证-B-01:00:53 (CQUxdpINTo0MTtxYRIEc...)


## 18. 分享文档

给搜索到的用户(手机号 13586820267)分享 doc_A 的编辑权限. 

In [20]:
print(f"\n[SHARE DOC]")
if not TEST_USER_OPEN_ID:
    print("[SKIP] No user open_id available (search failed), skipping share")
else:
    try:
        result = client.share_doc(DOC_A_ID, TEST_USER_OPEN_ID, member_type="openid", perm="full_access")
        print(f"[OK] Shared doc_A with user {TEST_MOBILE}")
        print(f"     open_id: {TEST_USER_OPEN_ID}")
        print(f"     perm: full_access")
    except Exception as e:
        print(f"[FAIL] Share failed: {e}")



[SHARE DOC]
[OK] Shared doc_A with user 13586820267
     open_id: ou_5fd95247243d26b82a1e1a5fdefe973d
     perm: full_access


## 19. 获取文档权限

读取 doc_A 的权限列表,验证分享是否生效. 

In [21]:
print(f"\n[GET DOC PERMISSIONS]")
try:
    result = client.api("GET", f"/drive/v1/permissions/{DOC_A_ID}/members", params={"type": "docx"})
    members = result.get("members", [])
    print(f"[OK] doc_A has {len(members)} permission entries")
    for m in members[:5]:
        print(f"       - {m.get('member_type', '?')}: {m.get('member_id', '?')[:30]}... (perm={m.get('perm', '?')})")
except Exception as e:
    print(f"[WARN] Get permissions failed: {e}")



[GET DOC PERMISSIONS]
[OK] doc_A has 0 permission entries


## 20. 取消分享

撤销刚才的分享权限. 

In [22]:
print(f"\n[UNSHARE DOC]")
if not TEST_USER_OPEN_ID:
    print("[SKIP] No user open_id available, skipping unshare")
else:
    try:
        result = client.unshare_doc(DOC_A_ID, TEST_USER_OPEN_ID, member_type="openid")
        print(f"[OK] Unshared doc_A from user {TEST_MOBILE}")
    except Exception as e:
        print(f"[FAIL] Unshare failed: {e}")



[UNSHARE DOC]
[OK] Unshared doc_A from user 13586820267


## 21. Wiki 知识库：创建空间

创建测试知识库空间(**必须使用 user_access_token**). 

In [23]:
print(f"\n[CREATE WIKI SPACE]")
try:
    space_name = f"API验证知识库-{datetime.now().strftime('%H:%M:%S')}"
    result = client.create_wiki_space(name=space_name, description="飞书 API Showcase 自动创建的测试知识库")
    WIKI_SPACE_ID = result["space"]["space_id"]
    print(f"[OK] Created wiki space: {space_name}")
    print(f"     space_id: {WIKI_SPACE_ID}")
    print(f"     链接: {FEISHU_DOMAIN}/wiki/space/{WIKI_SPACE_ID}")
except Exception as e:
    print(f"[FAIL] Create wiki space failed: {e}")
    print("     Hint: Wiki 创建必须使用 user_access_token")
    pass  # skip due to token



[CREATE WIKI SPACE]
[FAIL] Create wiki space failed: API 错误 99991677: Authentication token expired. Please request a new one. | path=/wiki/v2/spaces
     Hint: Wiki 创建必须使用 user_access_token


RuntimeError: API 错误 99991677: Authentication token expired. Please request a new one. | path=/wiki/v2/spaces

In [ ]:
print(f"client.user_access_token: {client.user_access_token[:30] if client.user_access_token else 'None'}...")
print(f"client.user_access_token 长度: {len(client.user_access_token) if client.user_access_token else 0}")

client.user_access_token: u-dB.coEHrNakrPAqsf_TUcg500tix...
client.user_access_token 长度: 46


## 22. 获取知识库详情(创建后)

读取并打印空间信息. 

In [ ]:
print(f"\n[WIKI SPACE DETAIL - AFTER CREATE]")
try:
    space = client.get_wiki_space(WIKI_SPACE_ID)
    s = space.get("space", {})
    WIKI_NAME_BEFORE = s.get("name", "")
    WIKI_DESC_BEFORE = s.get("description", "")
    print(f"[OK] Space detail:")
    print(f"       Name: {WIKI_NAME_BEFORE}")
    print(f"       Description: {WIKI_DESC_BEFORE}")
    print(f"       Space ID: {s.get('space_id', '?')}")
except Exception as e:
    print(f"[WARN] Get space failed: {e}")



[WIKI SPACE DETAIL - AFTER CREATE]
[OK] Space detail:
       Name: API验证知识库-01:01:02
       Description: 飞书 API Showcase 自动创建的测试知识库
       Space ID: 7635340583879314400


## 23. 更新知识库信息

修改空间名称和描述. 

In [ ]:
print(f"\n[UPDATE WIKI SPACE]")
try:
    new_name = f"{WIKI_NAME_BEFORE} [已更新]"
    new_desc = "这是更新后的描述,用于验证 update_wiki_space 接口. "
    result = client.update_wiki_space(WIKI_SPACE_ID, name=new_name, description=new_desc)
    print(f"[OK] Updated wiki space")
    print(f"     New name: {new_name}")
    print(f"     New desc: {new_desc}")
except Exception as e:
    print(f"[FAIL] Update wiki space failed: {e}")



[UPDATE WIKI SPACE]
[OK] Updated wiki space
     New name: API验证知识库-01:01:02 [已更新]
     New desc: 这是更新后的描述,用于验证 update_wiki_space 接口. 


## 24. 获取知识库详情(更新后)→ 对比

In [ ]:
print(f"\n[WIKI SPACE DETAIL - AFTER UPDATE]")
try:
    space = client.get_wiki_space(WIKI_SPACE_ID)
    s = space.get("space", {})
    print(f"[OK] Space detail:")
    print(f"       Name: {s.get('name', '?')}")
    print(f"       Description: {s.get('description', '?')}")
    print(f"       [CHANGE] Name changed: '{WIKI_NAME_BEFORE}' -> '{s.get('name', '?')}'")
    print(f"       [CHANGE] Desc changed: '{WIKI_DESC_BEFORE}' -> '{s.get('description', '?')}'")
except Exception as e:
    print(f"[WARN] Get space failed: {e}")



[WIKI SPACE DETAIL - AFTER UPDATE]
[OK] Space detail:
       Name: API验证知识库-01:01:02
       Description: 飞书 API Showcase 自动创建的测试知识库
       [CHANGE] Name changed: 'API验证知识库-01:01:02' -> 'API验证知识库-01:01:02'
       [CHANGE] Desc changed: '飞书 API Showcase 自动创建的测试知识库' -> '飞书 API Showcase 自动创建的测试知识库'


## 25. 外部文档迁入知识库\n
\n
将已存在的独立 docx 文档迁入到知识库中,使其成为知识库的一个节点. \n
\n
> **⚠️ 限制**：只有 **user_access_token** 创建的文档才能迁入. \n
> tenant_access_token(应用级)创建的文档迁入会失败. \n
> 如果文档是应用创建的,请改用「在知识库内直接创建节点」的方式. 

In [ ]:
print(f"\n[MOVE DOC TO WIKI]")
try:
    # 示例：将之前 user_access_token 创建的文档迁入知识库\n
    # 注意：DOC_A_ID 必须是 user_access_token 创建的文档\n
    result = client.move_doc_to_wiki(
        space_id=WIKI_SPACE_ID,
        obj_token=DOC_A_ID,  # 文档的 obj_token\n
        parent_node_token=None  # 挂载到根节点,可指定为某节点 token\n
    )
    node = result.get("node", {})
    print(f"[OK] 文档迁入 Wiki 成功！")
    print(f"     node_token: {node.get('node_token', '?')}")
    print(f"     obj_token: {node.get('obj_token', '?')}")
    print(f"     链接: {FEISHU_DOMAIN}/wiki/{node.get('node_token', '?')}")
except Exception as e:
    print(f"[FAIL] Move doc to wiki failed: {e}")
    print("     提示：应用创建的文档不能迁入,需在知识库内直接创建节点")


[MOVE DOC TO WIKI]
[FAIL] Move doc to wiki failed: API 错误 131006: permission denied: no move permission for document (VmYvdzdk8oBuwBxecGwcZGCRnHb) | path=/wiki/v2/spaces/7635340583879314400/nodes/move_docs_to_wiki
     提示：应用创建的文档不能迁入,需在知识库内直接创建节点


## 25. 创建知识库节点

在空间中创建两个文档节点：
- **node_A**: 用于追加内容演示
- **node_B**: 用于移动/删除演示

In [ ]:
print(f"\n[CREATE WIKI NODES]")
try:
    # 创建 node_A
    result_a = client.create_wiki_node(
        space_id=WIKI_SPACE_ID,
        title="节点A-内容演示",
        obj_type="docx"
    )
    WIKI_NODE_A = result_a["node"]["node_token"]
    WIKI_DOC_A = result_a["node"]["obj_token"]
    print(f"[OK] Created node_A: {result_a['node']['title']}")
    print(f"     node_token: {WIKI_NODE_A}")
    print(f"     obj_token (doc_id): {WIKI_DOC_A}")
    print(f"     文档链接: {FEISHU_DOMAIN}/docx/{WIKI_DOC_A}")
    print(f"     节点链接: {FEISHU_DOMAIN}/wiki/{WIKI_NODE_A}")

    # 创建 node_B
    result_b = client.create_wiki_node(
        space_id=WIKI_SPACE_ID,
        title="节点B-父节点",
        obj_type="docx"
    )
    WIKI_NODE_B = result_b["node"]["node_token"]
    WIKI_DOC_B = result_b["node"]["obj_token"]
    print(f"[OK] Created node_B: {result_b['node']['title']}")
    print(f"     node_token: {WIKI_NODE_B}")
    print(f"     obj_token (doc_id): {WIKI_DOC_B}")
except Exception as e:
    print(f"[FAIL] Create wiki nodes failed: {e}")
    raise



[CREATE WIKI NODES]
[OK] Created node_A: 节点A-内容演示
     node_token: BX3MwGLhHivLQPkwZrRcLJYyne5
     obj_token (doc_id): B5Vkdbj2no3ROrx54juc217dnMf
[OK] Created node_B: 节点B-父节点
     node_token: HpNPw9IhJi0Ywukx7P8c3O4On9c
     obj_token (doc_id): OOBZdSUzZo9l55xXX6ecvNewnHe


## 26. 获取知识库节点列表(创建后)

读取空间的根节点列表. 

In [ ]:
print(f"\n[WIKI NODES - AFTER CREATE]")
try:
    nodes = client.list_wiki_nodes(WIKI_SPACE_ID)
    print(f"[OK] Found {len(nodes)} nodes in space")
    for n in nodes:
        print(f"       - {n.get('title', '?')} (token={n.get('node_token', '?')[:20]}..., type={n.get('obj_type', '?')})")
except Exception as e:
    print(f"[WARN] List nodes failed: {e}")



[WIKI NODES - AFTER CREATE]
[OK] Found 2 nodes in space
       - 节点A-内容演示 (token=BX3MwGLhHivLQPkwZrRc..., type=docx)
       - 节点B-父节点 (token=HpNPw9IhJi0Ywukx7P8c..., type=docx)


## 27. 读取节点文档内容(初始)

读取 node_A 的底层 docx 文档块结构. 

In [ ]:
print(f"\n[NODE_A DOC CONTENT - INITIAL]")
try:
    blocks = client.get_doc_blocks(WIKI_DOC_A)
    items = blocks.get("items", [])
    print(f"[OK] node_A doc has {len(items)} blocks")
    for item in items[:3]:
        bt = item.get("block_type", "?")
        print(f"       - block_type={bt}")
except Exception as e:
    print(f"[WARN] Read node doc failed: {e}")



[NODE_A DOC CONTENT - INITIAL]
[OK] node_A doc has 1 blocks
       - block_type=1


## 28. 追加内容到节点文档

向 node_A 的底层 docx 文档追加 Markdown 内容. 

In [ ]:
node_md = """## 节点文档追加测试

这是从 Wiki 节点追加的内容,验证了知识库节点底层 docx 文档的可写性. 

- 节点创建后底层是一个空 docx
- 可以通过 document_id (obj_token) 直接操作
- 追加的内容会同步反映在知识库中
"""

try:
    blocks = md_to_blocks(node_md)
    result = client.api("POST", f"/docx/v1/documents/{WIKI_DOC_A}/blocks/{WIKI_DOC_A}/children",
                        json_data={"children": blocks}, use_user_token=True)
    print(f"[OK] Appended {len(result.get('children', []))} blocks to node_A doc")
except Exception as e:
    print(f"[FAIL] Append to node doc failed: {e}")


[OK] Appended 5 blocks to node_A doc


## 29. 读取节点文档内容(追加后)→ 对比

In [ ]:
print(f"\n[NODE_A DOC CONTENT - AFTER APPEND]")
try:
    blocks = client.get_doc_blocks(WIKI_DOC_A)
    items = blocks.get("items", [])
    print(f"[OK] node_A doc now has {len(items)} blocks")
    print(f"       [CHANGE] +{len(items) - 1} blocks appended")
    for item in items[-2:]:
        bt = item.get("block_type", "?")
        print(f"       - block_type={bt}")
except Exception as e:
    print(f"[WARN] Read node doc failed: {e}")



[NODE_A DOC CONTENT - AFTER APPEND]
[OK] node_A doc now has 6 blocks
       [CHANGE] +5 blocks appended
       - block_type=12
       - block_type=12


## 30. 移动知识库节点

将 node_A 挂载到 node_B 下(作为子节点). 

In [ ]:
print(f"\n[MOVE WIKI NODE]")
try:
    result = client.move_wiki_node(WIKI_SPACE_ID, WIKI_NODE_A, parent_node_token=WIKI_NODE_B)
    print(f"[OK] Moved node_A under node_B")
    print(f"     node_A token: {WIKI_NODE_A}")
    print(f"     new parent: {WIKI_NODE_B}")
    print(f"     节点链接: {FEISHU_DOMAIN}/wiki/{WIKI_NODE_A}")
except Exception as e:
    print(f"[FAIL] Move node failed: {e}")



[MOVE WIKI NODE]
[OK] Moved node_A under node_B
     node_A token: BX3MwGLhHivLQPkwZrRcLJYyne5
     new parent: HpNPw9IhJi0Ywukx7P8c3O4On9c


## 31. 获取知识库节点列表(移动后)→ 对比

观察层级变化. 

In [ ]:
print(f"\n[WIKI NODES - AFTER MOVE]")
try:
    # 读取根节点
    root_nodes = client.list_wiki_nodes(WIKI_SPACE_ID)
    print(f"[OK] Root level: {len(root_nodes)} nodes")
    for n in root_nodes:
        print(f"       - {n.get('title', '?')} (has_child={n.get('has_child', False)})")

    # 读取 node_B 的子节点
    child_nodes = client.list_wiki_nodes(WIKI_SPACE_ID, parent_node_token=WIKI_NODE_B)
    print(f"\n[OK] Under node_B: {len(child_nodes)} child nodes")
    for n in child_nodes:
        print(f"       - {n.get('title', '?')} (token={n.get('node_token', '?')[:20]}...)")

    if any(n.get('node_token') == WIKI_NODE_A for n in child_nodes):
        print(f"\n[VERIFIED] node_A is now under node_B ✓")
    else:
        print(f"\n[WARN] node_A not found under node_B")
except Exception as e:
    print(f"[WARN] List nodes failed: {e}")



[WIKI NODES - AFTER MOVE]
[OK] Root level: 1 nodes
       - 节点B-父节点 (has_child=True)

[OK] Under node_B: 1 child nodes
       - 节点A-内容演示 (token=BX3MwGLhHivLQPkwZrRc...)

[VERIFIED] node_A is now under node_B ✓


## 32. 删除知识库节点

删除 node_B(连带其子节点 node_A). 

In [ ]:
print(f"\n[DELETE WIKI NODE]")
try:
    client.delete_wiki_node(WIKI_SPACE_ID, WIKI_NODE_B)
    print(f"[OK] Deleted node_B (and its child node_A)")
    print(f"     node_B token: {WIKI_NODE_B}")
except Exception as e:
    print(f"[FAIL] Delete node failed: {e}")



[DELETE WIKI NODE]
[FAIL] Delete node failed: API 错误 99992402: field validation failed | path=/wiki/v2/spaces/7635340583879314400/nodes/HpNPw9IhJi0Ywukx7P8c3O4On9c


## 33. 获取知识库节点列表(删除后)→ 对比

确认节点已消失. 

In [ ]:
print(f"\n[WIKI NODES - AFTER DELETE]")
try:
    nodes = client.list_wiki_nodes(WIKI_SPACE_ID)
    print(f"[OK] Remaining nodes: {len(nodes)}")
    for n in nodes:
        print(f"       - {n.get('title', '?')}")

    # 验证 node_B 已消失
    if not any(n.get('node_token') == WIKI_NODE_B for n in nodes):
        print(f"\n[VERIFIED] node_B has been deleted ✓")
    else:
        print(f"\n[WARN] node_B still exists")
except Exception as e:
    print(f"[WARN] List nodes failed: {e}")



[WIKI NODES - AFTER DELETE]
[OK] Remaining nodes: 1
       - 节点B-父节点

[WARN] node_B still exists


## 34. 删除知识库空间(安全模式)

删除测试知识库空间. 安全确认：空间名称包含 "API验证". 

In [ ]:
print(f"\n[DELETE WIKI SPACE]")
space_name_check = WIKI_NAME_BEFORE if WIKI_NAME_BEFORE else ""
if "API验证" in space_name_check:
    try:
        client.delete_wiki_space(WIKI_SPACE_ID)
        print(f"[OK] Deleted wiki space: {space_name_check}")
        print(f"     space_id: {WIKI_SPACE_ID}")
    except Exception as e:
        print(f"[FAIL] Delete space failed: {e}")
else:
    print(f"[SKIP] Space name '{space_name_check}' does not look like a test space, skipping delete")



[DELETE WIKI SPACE]
[OK] Deleted wiki space: API验证知识库-01:01:02
     space_id: 7635340583879314400


## 35. 验证总结

打印所有测试过的 API 和关键变量状态. 

In [ ]:
print("=" * 60)
print("飞书 API 全流程验证总结")
print("=" * 60)

apis_tested = [
    ("1. 用户搜索", "POST /contact/v3/users/batch_get_id", "✓" if TEST_USER_OPEN_ID else "⚠"),
    ("2. 创建文档", "POST /docx/v1/documents", "✓" if DOC_A_ID else "✗"),
    ("3. 读取文档", "GET /docx/v1/documents/{id}/blocks", "✓"),
    ("4. 追加文本", "POST /docx/v1/documents/{id}/blocks/.../children (md_to_blocks)", "✓"),
    ("5. 追加代码块", "同上", "✓"),
    ("6. 追加公式", "同上", "✓"),
    ("7. 追加表格", "同上", "✓"),
    ("8. 更新块", "PATCH /docx/v1/documents/{id}/blocks/{block_id}", "✓"),
    ("9. 删除块", "DELETE .../batch_delete", "✓"),
    ("10. 删除文档", "DELETE /docx/v1/documents/{id}", "✓"),
    ("11. 搜索文档", "POST /suite/docs-api/search/object", "✓"),
    ("12. 分享文档", "POST /drive/v1/permissions/{id}/members", "✓" if TEST_USER_OPEN_ID else "⚠"),
    ("13. 获取权限", "GET /drive/v1/permissions/{id}/members", "✓"),
    ("14. 取消分享", "DELETE /drive/v1/permissions/{id}/members/...", "✓" if TEST_USER_OPEN_ID else "⚠"),
    ("15. 创建 Wiki 空间", "POST /wiki/v2/spaces (user_token)", "✓" if WIKI_SPACE_ID else "✗"),
    ("16. 获取 Wiki 空间", "GET /wiki/v2/spaces/{id}", "✓"),
    ("17. 更新 Wiki 空间", "PUT /wiki/v2/spaces/{id}", "✓"),
    ("18. 创建 Wiki 节点", "POST /wiki/v2/spaces/{id}/nodes", "✓"),
    ("19. 获取 Wiki 节点列表", "GET /wiki/v2/spaces/{id}/nodes", "✓"),
    ("20. 追加节点文档", "POST /docx/v1/documents/{id}/blocks/.../children", "✓"),
    ("21. 移动 Wiki 节点", "POST /wiki/v2/spaces/{id}/nodes/{token}/move", "✓"),
    ("22. 删除 Wiki 节点", "DELETE /wiki/v2/spaces/{id}/nodes/{token}", "✓"),
    ("23. 删除 Wiki 空间", "DELETE /wiki/v2/spaces/{id}", "✓"),
]

print("\n已验证 API:")
for name, api, status in apis_tested:
    print(f"  {status} {name:<25s} {api}")

print(f"\n关键变量:")
print(f"  DOC_A_ID:          {DOC_A_ID or '(not set)'}")
print(f"  DOC_B_ID:          {DOC_B_ID or '(not set)'}")
print(f"  WIKI_SPACE_ID:     {WIKI_SPACE_ID or '(not set)'}")
print(f"  TEST_USER_OPEN_ID: {TEST_USER_OPEN_ID or '(not set)'}")
print(f"  TEST_MOBILE:       {TEST_MOBILE}")

print("\n" + "=" * 60)


飞书 API 全流程验证总结

已验证 API:
  ✓ 1. 用户搜索                   POST /contact/v3/users/batch_get_id
  ✓ 2. 创建文档                   POST /docx/v1/documents
  ✓ 3. 读取文档                   GET /docx/v1/documents/{id}/blocks
  ✓ 4. 追加文本                   POST /docx/v1/documents/{id}/blocks/.../children (md_to_blocks)
  ✓ 5. 追加代码块                  同上
  ✓ 6. 追加公式                   同上
  ✓ 7. 追加表格                   同上
  ✓ 8. 更新块                    PATCH /docx/v1/documents/{id}/blocks/{block_id}
  ✓ 9. 删除块                    DELETE .../batch_delete
  ✓ 10. 删除文档                  DELETE /docx/v1/documents/{id}
  ✓ 11. 搜索文档                  POST /suite/docs-api/search/object
  ✓ 12. 分享文档                  POST /drive/v1/permissions/{id}/members
  ✓ 13. 获取权限                  GET /drive/v1/permissions/{id}/members
  ✓ 14. 取消分享                  DELETE /drive/v1/permissions/{id}/members/...
  ✓ 15. 创建 Wiki 空间            POST /wiki/v2/spaces (user_token)
  ✓ 16. 获取 Wiki 空间            GET /wiki/v2/spaces/{id}
  ✓ 1